# Clustering teks · toolkit modular v2

Vocabulary dan topic ID dibekukan. Baca contoh dokumen per topik; metrik internal tidak mengukur akurasi sentimen.

Ubah konfigurasi. Hapus/tambah blok melalui perakit, bukan mengubah puluhan sel. Semua model menggunakan split/urutan label yang sama. Hasil harus dihitung ulang di Kaggle; skor notebook sumber tidak diwariskan.

Sumber adaptasi: `notebook/a_tanpa_label/03_clustering/03_clustering_review.ipynb`.

## 1 · Konfigurasi

File, kolom, preprocessing dan model hanya di sini. Group ID pengguna opsional. Untuk offline, set offline=True dan ganti model path ke bundle Kaggle input.

In [ ]:
CFG = {'data_file': '/kaggle/input/GANTI_DATASET/train.csv',
 'encoding': 'utf-8',
 'text_col': 'text',
 'group_col': None,
 'seed': 42,
 'out': '/kaggle/working/hasil_toolkit',
 'offline': False,
 'preprocess': 'bow',
 'slang': {},
 'stopwords': [],
 'eda_sample': 5000,
 'eda_top': 15,
 'eda_languages': False,
 'max_features': 60000,
 'max_len': 192,
 'e5_model': 'intfloat/multilingual-e5-base',
 'k_values': [4, 6, 8],
 'fit_rows': 5000,
 'task': 'clustering',
 'models': ['nmf', 'lda', 'kmeans'],
 'sentiment_pretrained': False}

## 2 · Lingkungan Kaggle

Pasang hanya dependensi komponen terpilih. Untuk bundle offline: rakit dengan --no-install atau hapus sel instalasi.

In [ ]:
import os, sys, subprocess
from pathlib import Path
assert Path('/kaggle/input').exists(), 'Jalankan di Kaggle.'
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'emoji>=2,<3', 'ftfy>=6.3,<7', 'numpy', 'pandas', 'scikit-learn', 'scipy'], check=True)

## 3 · Fungsi komponen

Definisi sekali; sel pemakaian pendek berada di bawah. Semua dependensi disertakan, tanpa import dari folder toolkit.

### Kontrak, data loading & penyimpanan



In [ ]:
"""Kontrak bersama. Tidak memasang paket atau menjalankan model saat disalin."""
import json
import random
import re
import time
from pathlib import Path
import numpy as np
import pandas as pd

MODEL_RUNNERS = {}
ENCODERS = {}


def encode_e5(texts, cfg, role='query'):
    assert 'e5' in ENCODERS, 'Pasang komponen encoder E5 melalui perakit.'
    return ENCODERS['e5'](texts,cfg,role)


def register(name):
    def decorate(fn):
        MODEL_RUNNERS[name] = fn
        return fn
    return decorate


def read_table(path, encoding="utf-8", string_cols=()):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Isi CFG['data_file'] dengan file Kaggle yang ada: {path}")
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    return pd.read_csv(path, sep="\t" if path.suffix == ".tsv" else ",", encoding=encoding,
        dtype={col:str for col in string_cols} or None)


def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)


def target_array(frame, cfg):
    return frame[cfg['label_cols']].to_numpy(float) if cfg['task']=='multilabel' else frame.y.to_numpy()


def require_kaggle(gpu=False):
    assert Path('/kaggle/input').exists(), "Jalankan eksperimen di Kaggle."
    if gpu:
        import torch
        assert torch.cuda.is_available(), "Aktifkan GPU Kaggle."


def show_table(name, table):
    from IPython.display import display
    print("\n" + name)
    display(table.to_frame() if isinstance(table, pd.Series) else table)


def save_tables(tables, out, prefix=""):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_csv(out / f"{prefix}{name}.csv", index=True)


### Preprocessing native / normalized / BoW / IndoBERTweet

Slang hanya dari CFG["slang"]. Native mempertahankan negasi, emoji, angka, case. Uji normalisasi sebagai eksperimen.

In [ ]:
import html
import unicodedata

NEGATIONS = {'tidak', 'tak', 'bukan', 'belum', 'jangan', 'tanpa', 'gak', 'nggak', 'ga', 'not', 'never'}


def safe_text(value):
    """Missing tetap menjadi kosong; angka, emoji, negasi, tanda baca dipertahankan."""
    if not isinstance(value, str):
        return ''
    import ftfy
    text = ftfy.fix_text(value, unescape_html=False, uncurl_quotes=False,
        fix_latin_ligatures=False, fix_character_width=False, remove_control_chars=False, normalization=None)
    text = unicodedata.normalize('NFC', html.unescape(text))
    text = text.translate(dict.fromkeys(map(ord, '\ufeff\u200b\u2060\x00'), None))
    text = re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def preprocess_text(value, preset='native', slang=None, stopwords=()):
    text = safe_text(value)
    if preset == 'native':
        return text
    if preset == 'indobertweet':
        import emoji
        text = text.lower()
        text = re.sub(r'https?://\S+|www\.\S+|\burl\b', 'HTTPURL', text)
        text = re.sub(r'(?<!\w)@\w+|\buser\b', '@USER', text)
        return safe_text(emoji.demojize(text))
    if preset not in {'normalized', 'bow'}:
        raise ValueError(f'Preset tidak dikenal: {preset}')
    text = text.casefold()
    if slang:
        text = re.sub(r'\b[^\W_]+\b', lambda m: slang.get(m[0], m[0]), text)
    if preset == 'bow':
        text = re.sub(r'https?://\S+|www\.\S+|@\w+', ' ', text)
        # Stopword diberikan pengguna, negasi selalu dilindungi; tidak ada stemming otomatis.
        blocked = set(stopwords) - NEGATIONS
        text = ' '.join(w for w in re.findall(r'[^\W\d_]+', text) if w not in blocked)
    return safe_text(text)


def make_cleaner(cfg):
    return lambda value: preprocess_text(value, cfg['preprocess'], cfg.get('slang'), cfg.get('stopwords', ()))


### EDA angka: missing → vocabulary → noise

Basic/noise seluruh baris; vocab/ngram memakai sampel eksplisit. Persentase noise boleh overlap. Bahasa adalah estimasi sampel.

In [ ]:
from collections import Counter

NOISE_PATTERNS = {
    'url': r'https?://|www\.', 'mention': r'@\w+', 'hashtag': r'#\w+', 'angka': r'\d',
    'emoji_simbol': r'[\U0001F300-\U0001FAFF\u2600-\u27BF]', 'huruf_ulang': r'([^\W\d_])\1{2,}',
    'baca_ulang': r'[!?.,]{2,}', 'html': r'</?[a-zA-Z][^>]*>', 'encoding': r'\ufffd|Ã.|Â.',
    'non_ascii': r'[^\x00-\x7f]', 'spasi_ulang': r' {2,}', 'invisible': r'[\u200b\ufeff\u2060]',
    'kapital': r'(?-i:\b[A-Z]{3,}\b)', 'huruf_angka': r'\b(?=\w*[a-z])(?=\w*\d)\w+\b',
    'slang_contoh': r'\b(?:gk|gak|nggak|ga|bgt|yg|utk|udh|klo|jg)\b',
    'negasi': r'\b(?:tidak|tak|bukan|belum|jangan|tanpa|gak|nggak|ga|not|never)\b'
}


def numeric_eda(frame, text_col, label_cols=(), top=15, sample=5000, seed=42, languages=False):
    """Hitungan dasar seluruh baris; vocabulary/ngram pada sampel yang ukurannya dilaporkan."""
    s = frame[text_col]
    text = s.where(s.map(lambda x: isinstance(x, str)), '').fillna('')
    nonempty = text.str.strip().ne('')
    pseudo = text.str.strip().str.casefold().isin({'nan', 'none', 'null', 'n/a', 'na', '-', '--'})
    quality = pd.Series({'rows': len(s), 'missing': s.isna().sum(), 'blank': (~nonempty).sum(),
        'pseudo_null': pseudo.sum(), 'non_string_nonmissing': (s.notna() & ~s.map(lambda x: isinstance(x, str))).sum(),
        'unique_nonempty': text[nonempty].nunique(), 'duplicate_extra_nonempty': text[nonempty].duplicated().sum(),
        'duplicate_members_nonempty': text[nonempty].duplicated(keep=False).sum()})
    subset = text[nonempty].sample(min(int(sample), int(nonempty.sum())), random_state=seed)
    tokens = subset.str.casefold().str.findall(r'[^\W\d_]+')
    counts = pd.Series(Counter(w for row in tokens for w in row), dtype='int64').sort_values(ascending=False)
    n_tokens = int(counts.sum())
    tables = {'quality': quality.to_frame('n'),
        'missing_unique': pd.DataFrame({'missing': frame.isna().sum(), 'missing_pct': frame.isna().mean()*100,
            'unique': frame.apply(lambda col:col.map(lambda value:json.dumps(value,sort_keys=True) if isinstance(value,(list,dict)) else value).nunique())}),
        'length': pd.DataFrame({'char': text.str.len(), 'word': text.str.split().str.len()}).describe(percentiles=[.01,.5,.9,.95,.99]).T,
        'vocabulary': pd.Series({'sample_rows': len(subset), 'tokens': n_tokens, 'types': len(counts),
            'hapax_types': counts.eq(1).sum(), 'hapax_pct_types': counts.eq(1).mean()*100,
            'hapax_pct_tokens': counts.eq(1).sum()/max(n_tokens,1)*100}).to_frame('value'),
        'top_words': counts.head(top).to_frame('n').assign(pct_tokens=lambda d: d.n/max(n_tokens,1)*100)}
    for n in (2,3):
        grams = Counter(' '.join(row[i:i+n]) for row in tokens for i in range(len(row)-n+1))
        tables[f'ngram_{n}'] = pd.DataFrame(grams.most_common(top), columns=['term', 'n'])
    flags = pd.DataFrame({name: text.str.count(pattern, flags=re.I).gt(0) for name,pattern in NOISE_PATTERNS.items()})
    tables['noise'] = pd.DataFrame({'n_rows': flags.sum(), 'pct_all_rows_overlap_allowed': flags.mean()*100})
    tables['odd_words_candidates'] = counts[[len(w)>=20 or bool(re.search(r'([^\W\d_])\1{2,}',w)) for w in counts.index]].head(top).to_frame('n')
    for label in label_cols:
        known = frame[label].dropna()
        count = known.value_counts()
        tables[f'label_{label}'] = count.to_frame('n').assign(pct_known=lambda d:d.n/max(len(known),1)*100)
        tables[f'conflict_{label}'] = frame.assign(_key=text.str.casefold()).loc[nonempty].groupby('_key')[label].nunique().value_counts().to_frame('text_groups')
    if languages and len(subset):
        from langid.langid import LanguageIdentifier, model
        detector = LanguageIdentifier.from_modelstring(model, norm_probs=True)
        small = subset.sample(min(150,len(subset)),random_state=seed)
        pairs = small.map(lambda t: detector.classify(t[:3000]) if len(t.split())>=5 else ('uncertain',0))
        lang = pairs.map(lambda p:p[0] if p[1]>=.8 else 'uncertain').value_counts()
        tables['language_estimate_sample_only'] = lang.to_frame('n').assign(pct_sample=lang/len(small)*100)
    return tables


def multilabel_eda(frame, labels):
    y = frame[labels].astype(float)
    complete = y.dropna()
    co = pd.DataFrame(index=labels, columns=labels, dtype=float)
    for a in labels:
        for b in labels:
            known = y[[a,b]].notna().all(axis=1)
            co.loc[a,b] = (y.loc[known,a].eq(1)&y.loc[known,b].eq(1)).mean()*100
    return {'label_support': pd.DataFrame({'known':y.notna().sum(), 'unknown':y.isna().sum(),
                'positive':y.eq(1).sum(), 'negative':y.eq(0).sum(), 'positive_pct_known':y.mean()*100}),
        'cardinality_complete_rows': complete.sum(axis=1).value_counts().sort_index().to_frame('n'),
        'label_combinations_complete_rows': complete.value_counts().head(15).to_frame('n'),
        'cooccurrence_pct_pairwise_known': co}


def pair_eda(frame, a, b, target):
    left,right = frame[a].map(safe_text),frame[b].map(safe_text)
    pairs = pd.DataFrame({'a':left.str.casefold(),'b':right.str.casefold()})
    lset = pairs.a.map(lambda t:set(re.findall(r'[^\W\d_]+',t)))
    rset = pairs.b.map(lambda t:set(re.findall(r'[^\W\d_]+',t)))
    overlap = pd.Series([len(x&y)/max(len(x|y),1) for x,y in zip(lset,rset)])
    reversed_keys = set(zip(pairs.b,pairs.a))
    numbers = pd.Series([set(re.findall(r'\d+',x))==set(re.findall(r'\d+',y)) for x,y in zip(left,right)])
    detail = pd.DataFrame({'label_or_score':frame[target].to_numpy(),'jaccard':overlap,
        'abs_word_length_difference':(left.str.split().str.len()-right.str.split().str.len()).abs(),
        'identical':pairs.a.eq(pairs.b),'same_numbers':numbers})
    return {'pair_quality':pd.Series({'rows':len(frame),'identical_sides':pairs.a.eq(pairs.b).sum(),
            'duplicate_ordered_pairs_extra':pairs.duplicated().sum(),
            'pairs_with_reverse_present':sum((x,y) in reversed_keys for x,y in zip(pairs.a,pairs.b)),
            'shared_endpoints':len(set(pairs.a)&set(pairs.b))}).to_frame('n'),
        'pair_relation_statistics':detail.drop(columns='label_or_score').describe().T,
        'pair_relation_by_target':detail.groupby('label_or_score').agg(n=('jaccard','size'),mean_jaccard=('jaccard','mean'),
            mean_length_diff=('abs_word_length_difference','mean'),identical_pct=('identical',lambda v:v.mean()*100))}


def eda_for_task(frame, cfg):
    task = cfg['task'];tables={}
    columns = cfg['pair_cols'] if task in {'nli','similarity'} else [cfg.get('doc_text_col',cfg.get('text_col'))]
    labels = cfg['label_cols'] if task=='multilabel' else [cfg['label_col']] if task in {'classification','nli','similarity'} else []
    for i,col in enumerate(columns):
        part = numeric_eda(frame,col,labels if i==0 else (),cfg.get('eda_top',15),cfg['eda_sample'],cfg['seed'],cfg['eda_languages'])
        tables.update({str(col)+'_'+name:table for name,table in part.items()})
    if task in {'nli','similarity'}:
        tables.update(pair_eda(frame,*cfg['pair_cols'],cfg['label_col']))
    for name,table in tables.items():
        show_table(name,table)
    save_tables(tables,cfg['out'],'raw_eda_')
    return tables


### Grup pengguna, duplicate & connected endpoint



In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def connected_groups(frame, text_cols, group_col=None, normalized_cols=(), shared_endpoints=False):
    """Gabung pengguna, duplikat native, dan collision normalisasi sebelum split."""
    parent = np.arange(len(frame))
    def root(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    seen = {}
    for i,(_,row) in enumerate(frame.iterrows()):
        keys = [(('endpoint' if shared_endpoints else col), safe_text(row[col]).casefold()) for col in text_cols]
        keys += [('endpoint' if shared_endpoints else 'normalized_'+col, str(row[col]).casefold()) for col in normalized_cols]
        if group_col and pd.notna(row[group_col]):
            keys.append(('user',str(row[group_col])))
        for key in keys:
            if not key[1]:
                continue
            if key in seen:
                parent[root(i)] = root(seen[key])
            else:
                seen[key] = i
    return np.array([root(i) for i in range(len(frame))])



### Frozen topic/cluster fit + validation quality

Fit vocabulary/model pada train unik, pilih k di validation, audit holdout. NPMI/silhouette tidak menggantikan audit semantik.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.decomposition import NMF, LatentDirichletAllocation, TruncatedSVD
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score


def prepare_unsupervised(raw, cfg):
    frame = raw.copy().reset_index(drop=True)
    frame['_row_id'] = np.arange(len(frame))
    frame['native'] = frame[cfg['text_col']].map(safe_text)
    frame['a'] = frame[cfg['text_col']].map(make_cleaner(cfg))
    frame = frame[frame.a.ne('')].reset_index(drop=True)
    frame['_group'] = connected_groups(frame,[cfg['text_col']],cfg.get('group_col'),['a'])
    if cfg['task']=='temporal':
        source = frame[cfg['time_col']]
        def timestamp(value):
            try:
                t = pd.Timestamp(value)
                return t.tz_localize(cfg['naive_timezone']) if t.tzinfo is None else t
            except (ValueError,TypeError):
                return pd.NaT
        utc = pd.to_datetime(source.map(timestamp),utc=True,errors='coerce')
        frame['timestamp'] = utc.dt.tz_convert(cfg['timezone'])
        print('Tanggal invalid dibuang:',int(frame.timestamp.isna().sum()))
        frame = frame[frame.timestamp.notna()].sort_values('timestamp').reset_index(drop=True)
        frame['period'] = frame.timestamp.dt.tz_localize(None).dt.to_period(cfg['period']).dt.start_time
        periods = sorted(frame.period.unique())
        assert len(periods)>=5, 'Perlu minimal lima periode untuk early/middle/late'
        a,b = max(1,int(len(periods)*.6)),max(2,int(len(periods)*.8))
        parts = {'train':frame[frame.period.isin(periods[:a])], 'val':frame[frame.period.isin(periods[a:b])], 'test':frame[frame.period.isin(periods[b:])]}
        # Tidak membuang volume berulang. Audit recurrence: group overlap lintas waktu dilaporkan.
        for split in ['val','test']:
            print('Recurring exact-text rows vs early,',split,':',parts[split]._group.isin(parts['train']._group).sum())
    else:
        rest,test = next(GroupShuffleSplit(1,test_size=.2,random_state=cfg['seed']).split(frame,groups=frame._group))
        train,val = next(GroupShuffleSplit(1,test_size=.25,random_state=cfg['seed']+1).split(frame.iloc[rest],groups=frame.iloc[rest]._group))
        parts = {'train':frame.iloc[rest[train]],'val':frame.iloc[rest[val]],'test':frame.iloc[test]}
    parts = {k:v.reset_index(drop=True) for k,v in parts.items()}
    fit = parts['train'].drop_duplicates('a')
    if cfg.get('fit_rows'):
        fit = fit.sample(min(len(fit),cfg['fit_rows']),random_state=cfg['seed'])
    parts['fit'] = fit.reset_index(drop=True)
    assert all(len(parts[k]) for k in ['fit','val','test'])
    return parts


def npmi_quality(texts, topics):
    terms = sorted({word for topic in topics for word in topic})
    if not terms or not len(texts):
        return np.nan
    vec = CountVectorizer(vocabulary=terms,binary=True)
    x = vec.transform(texts).astype('float64')
    prob = np.asarray(x.mean(0)).ravel()
    joint = (x.T@x).toarray()/len(texts)
    vocab = vec.vocabulary_;scores=[]
    for topic in topics:
        values = []
        for i,a in enumerate(topic):
            for b in topic[i+1:]:
                pa,pb,pab = prob[vocab[a]],prob[vocab[b]],joint[vocab[a],vocab[b]]
                values.append(np.log(pab/(pa*pb))/-np.log(pab) if 0<pab<1 and pa*pb>0 else (1. if pab==1 else -1.))
        if values:
            scores.append(np.mean(values))
    return float(np.mean(scores)) if scores else np.nan


def fit_topic_candidate(data, cfg, method, k):
    from scipy.sparse import csr_matrix
    assert len(data['fit'])>k, 'Teks unik train terlalu sedikit; kurangi k atau tambah data.'
    vec = CountVectorizer(min_df=2,max_df=1.,max_features=cfg['max_features'])
    counts = vec.fit_transform(data['fit'].a)
    assert min(counts.shape)>k, 'Kurangi k atau tambah ukuran train/vocabulary'
    tfidf = TfidfTransformer().fit(counts)
    dense = method=='e5_kmeans'
    reducer = None
    if method=='nmf':
        x = tfidf.transform(counts)
        model = NMF(k,init='nndsvda',max_iter=300,random_state=cfg['seed']).fit(x)
    elif method=='lda':
        x = counts
        model = LatentDirichletAllocation(k,max_iter=10,learning_method='batch',random_state=cfg['seed']).fit(x)
    else:
        if dense:
            x = encode_e5(data['fit'].native,cfg,'query')
        else:
            n = min(100,min(counts.shape)-1)
            reducer = TruncatedSVD(n,random_state=cfg['seed']).fit(tfidf.transform(counts))
            x = normalize(reducer.transform(tfidf.transform(counts)))
        model = MiniBatchKMeans(k,n_init=10,batch_size=512,random_state=cfg['seed']).fit(x)
    def represent(frame):
        if dense:
            return encode_e5(frame.native,cfg,'query')
        c = vec.transform(frame.a)
        if method=='lda':
            return c
        return normalize(reducer.transform(tfidf.transform(c))) if reducer is not None else tfidf.transform(c)
    def assign(frame):
        z = represent(frame)
        return model.transform(z).argmax(1) if method in {'nmf','lda'} else model.predict(z)
    if method in {'nmf','lda'}:
        weights = model.components_
    else:
        grouped = np.array([np.asarray(counts[model.labels_==j].sum(0)).ravel() for j in range(k)])
        weights = TfidfTransformer().fit_transform(csr_matrix(grouped)).toarray()
    words = np.array(vec.get_feature_names_out())
    topics = [words[np.argsort(-row)[:10]].tolist() for row in weights]
    return {'model':model,'assign':assign,'represent':represent,'topics':topics,'vectorizer':vec,'k':k,'method':method}


def run_unsupervised(data, cfg):
    candidates = {};rows=[]
    for method in cfg['models']:
        for k in cfg['k_values']:
            fitted = fit_topic_candidate(data,cfg,method,k)
            name = f'{method}_k{k}';candidates[name]=fitted
            val_ids = fitted['assign'](data['val'])
            row = {'model':name,'val_npmi':npmi_quality(data['val'].a,fitted['topics']),
                'val_active_groups':len(np.unique(val_ids)), 'topic_diversity':len(set(sum(fitted['topics'],[])))/sum(map(len,fitted['topics']))}
            if cfg['task']=='clustering':
                z = fitted['represent'](data['val'])
                row['val_silhouette'] = silhouette_score(z,val_ids,sample_size=min(1000,len(val_ids)),random_state=cfg['seed']) if 1<len(np.unique(val_ids))<len(val_ids) else np.nan
            rows.append(row)
    board = pd.DataFrame(rows)
    metric = 'val_silhouette' if cfg['task']=='clustering' else 'val_npmi'
    assert board[metric].notna().any(), f'Tidak ada kandidat valid untuk {metric}'
    chosen = board.sort_values(metric,ascending=False,kind='stable').iloc[0].model
    fitted = candidates[chosen]
    out = Path(cfg['out']);out.mkdir(parents=True,exist_ok=True)
    (out/'selection_frozen_before_late.json').write_text(json.dumps({'model':chosen,'selection_metric':metric}))
    assignments = pd.concat([data[s].assign(split=s,topic=fitted['assign'](data[s])) for s in ['train','val','test']],ignore_index=True)
    topics = pd.DataFrame({'topic':range(fitted['k']),'words':[' | '.join(t) for t in fitted['topics']]})
    examples = assignments.groupby(['split','topic'],group_keys=False).head(5)
    tables = {'validation':board,'topics':topics,'examples_for_human_audit':examples,
        'assignments':assignments,'late_quality':pd.Series({'NPMI':npmi_quality(data['test'].a,fitted['topics'])}).to_frame('value')}
    save_tables(tables,out)
    for name in ['validation','topics','late_quality']:
        show_table(name,tables[name])
    print('FINAL:',chosen,'| NPMI/silhouette bukan bukti kebenaran semantik; audit contoh per topik/cluster.')
    return {'tables':tables,'assignments':assignments,'fitted':fitted}


## 4 · Jalankan

Holdout dibuka setelah pemilihan validation dibekukan. Selisih kecil perlu repeated group split. EDA angka tersedia sebelum modeling.

In [ ]:
require_kaggle(gpu=False)
seed_all(CFG['seed'])

### EDA raw

In [ ]:
raw = read_table(CFG['data_file'], CFG['encoding'])
tables = eda_for_task(raw, CFG)

### Frozen topic / cluster models

In [ ]:
data = prepare_unsupervised(raw, CFG)
result = run_unsupervised(data, CFG)